In [ ]:
from behavior_utils import get_fitted_model_names, get_fitted_latent

get_fitted_model_names(session_name='behavior_764769_2024-12-11_18-21-49.nwb')

In [ ]:
result=collect_behavior_model_summary(session_paths=['/root/capsule/data/behavior_nwb/behavior_764769_2024-12-11_18-21-49.nwb'])

In [ ]:
result['QLearning_L1F1_CK1_softmax_LPT']

In [ ]:
# run for all behavior sessions

import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bar
from behavior_qc_metrics_summary import collect_behavior_model_summary
from general_utils import find_ephys_sessions
from nwb_utils import NWBUtils
from behavior_utils import get_fitted_model_names, get_fitted_latent

# Define the root directory (based on the folder structure you provided)
root_dir = "/root/capsule/data/behavior_nwb/"

# Initialize a list to store the full paths of all .nwb files
nwb_files = []

# Walk through all directories and files under the root directory
for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.endswith(".nwb"):
            # Append the full path of each .nwb file
            nwb_files.append(os.path.join(root, file))




def _run_one_session(session_name: str) -> pd.DataFrame:
    df = collect_behavior_model_summary(session_paths=[session_name])
    if "session_name" not in df.columns:
        df = df.assign(session_name=session_name)
    return df

def collect_behavior_model_summary_parallel(
    session_paths,
    *,
    max_workers: int = 20,
) -> pd.DataFrame:
    dfs = []
    errors = []
    total_sessions = len(session_paths)
    
    # Initialize tqdm progress bar
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        fut_to_sess = {ex.submit(_run_one_session, s): s for s in session_paths}
        completed = 0  # Initialize the completed task counter
        
        # Wrap the as_completed in tqdm to show progress
        for fut in tqdm(as_completed(fut_to_sess), total=total_sessions, desc="Processing sessions", ncols=100):
            completed += 1  # Increment completed tasks count
            
            s = fut_to_sess[fut]
            
            try:
                dfs.append(fut.result())
            except Exception as e:
                errors.append((s, repr(e)))

    if errors:
        print("Some sessions failed:")
        for s, err in errors[:20]:
            print(f"  {s}: {err}")
        if len(errors) > 20:
            print(f"  ... plus {len(errors)-20} more")

    if not dfs:
        return pd.DataFrame()

    out = pd.concat(dfs, ignore_index=True, sort=False)

    if "session_name" in out.columns:
        out = out.sort_values(["session_name"]).reset_index(drop=True)

    return out

# Usage
summary = collect_behavior_model_summary_parallel(session_paths=nwb_files, max_workers=8)


# Save the summary to a CSV file in /root/capsule/scratch
output_path = '/root/capsule/scratch/behavior_model_summary_ephys_sessions.csv'
summary.to_csv(output_path, index=False)  # Save without the index column
print(f"Summary saved to {output_path}")




In [25]:
import importlib
import behavior_qc_visualization

importlib.reload(behavior_qc_visualization)

from behavior_qc_visualization import load_behavior_model_summary_csv

summary=load_behavior_model_summary_csv('/root/capsule/scratch/behavior_model_summary_ephys_sessions.csv')
#summary=load_behavior_model_summary_csv('/root/capsule/scratch/behavior_model_summary_general_behavior.csv')

In [29]:
MODEL = "QLearning_L1F1_CK1_softmax"   # <-- change to "QLearning_L1F1_CKfull_softmax" if needed
criteria = {
    f"{MODEL}_learn_rate": (lambda x: x > 0.15, "Learn Rate <= 0.15"),
    f"{MODEL}_biasL": (lambda x: np.abs(x) < 0.5, "Abs(Bias L) >= 0.5"),
    f"{MODEL}_LPT": (lambda x: x > 0.65, "LPT <= 0.65"),
    f"{MODEL}_reward_coefs_sum6": (lambda x: x < 0.8, "Reward Coefs Sum6 >= 0.8"),
    f"{MODEL}_deltaQ_entropy": (lambda x: x > 0.1, "DeltaQ Entropy <= 2.5"),
    f"{MODEL}_rpe_entropy": (lambda x: x > 0.1, "RPE Entropy <= 2.25"),
    f"{MODEL}_deltaQ_std": (lambda x: x > 0.1, "DeltaQ Std <= 0.2"),
    f"{MODEL}_sumQ_std": (lambda x: x > 0.1, "SumQ Std <= 0.15"),
    f"{MODEL}_rpe_std": (lambda x: x > 0.1, "RPE Std <= 0.45"),
}


In [30]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Configure which model's columns you want to filter on
# ---------------------------------------------------------


# ---------------------------------------------------------
# Define criteria (value must PASS the lambda to be satisfied)
# NaNs are treated as FAIL (and will be reported explicitly)
# ---------------------------------------------------------


# Optional: sanity check required columns exist
missing = [c for c in criteria.keys() if c not in summary.columns]
if missing:
    raise KeyError(
        "Missing required columns in summary:\n"
        + "\n".join(missing)
        + f"\n\nTip: check MODEL='{MODEL}' matches your column prefixes."
    )

# ---------------------------------------------------------
# Build mask (pass all criteria). NaN => fail.
# ---------------------------------------------------------
mask = np.ones(len(summary), dtype=bool)
for col, (fn, _) in criteria.items():
    mask &= summary[col].apply(lambda v: False if pd.isna(v) else bool(fn(v)))

satisfied_sessions = summary[mask]
not_satisfied_sessions = summary[~mask]

# ---------------------------------------------------------
# Explain failures
# ---------------------------------------------------------
def get_failed_reasons(row: pd.Series) -> str:
    reasons = []
    for col, (fn, msg) in criteria.items():
        v = row.get(col, np.nan)
        if pd.isna(v):
            reasons.append(f"{col} is NaN")
        elif not fn(v):
            reasons.append(msg)
    return ", ".join(reasons) if reasons else "None"

# ---------------------------------------------------------
# Printing / displaying (use display in notebooks if stdout is broken)
# ---------------------------------------------------------
cols_to_print = ["session_name"] + list(criteria.keys())

print(f"MODEL = {MODEL}")
print(f"Num satisfied: {len(satisfied_sessions)}")
print(f"Num not satisfied: {len(not_satisfied_sessions)}")

print("\nSessions that satisfy the criteria:")
for _, row in satisfied_sessions.iterrows():
    parts = [f"Session Name: {row.get('session_name', 'NA')}"]
    for col in criteria.keys():
        parts.append(f"{col.split(MODEL+'_', 1)[-1]}: {row.get(col, 'NA')}")
    print(", ".join(parts))

print("\nSessions that do not satisfy the criteria:")
for _, row in not_satisfied_sessions.iterrows():
    failed_reasons = get_failed_reasons(row)
    parts = [f"Session Name: {row.get('session_name', 'NA')}"]
    for col in criteria.keys():
        parts.append(f"{col.split(MODEL+'_', 1)[-1]}: {row.get(col, 'NA')}")
    parts.append(f"Failed Reason: {failed_reasons}")
    print(", ".join(parts))



MODEL = QLearning_L1F1_CK1_softmax
Num satisfied: 26
Num not satisfied: 18

Sessions that satisfy the criteria:
Session Name: /root/capsule/data/behavior_nwb/764787_2024-12-10_13-42-03.nwb, learn_rate: 0.2642226785006654, biasL: 0.2010418107169309, LPT: 0.7568636613601546, reward_coefs_sum6: 0.2451734098879532, deltaQ_entropy: 2.952807529712785, rpe_entropy: 2.9946005043910717, deltaQ_std: 0.4259828994581221, sumQ_std: 0.3158840865912353, rpe_std: 0.4961431121030871
Session Name: /root/capsule/data/behavior_nwb/764790_2024-12-18_12-14-41.nwb, learn_rate: 0.1500795855343441, biasL: -0.1764025433176872, LPT: 0.8029770398486618, reward_coefs_sum6: 0.4930956172684929, deltaQ_entropy: 2.969460831482356, rpe_entropy: 2.913725756984447, deltaQ_std: 0.3381197349910092, sumQ_std: 0.204630036388403, rpe_std: 0.5066009090691534
Session Name: /root/capsule/data/behavior_nwb/764791_2025-01-14_12-04-05.nwb, learn_rate: 0.1700083124495701, biasL: -0.2806146779177897, LPT: 0.7322000491060761, reward_c

In [21]:
import numpy as np
import pandas as pd


# Sanity check required columns exist
missing = [c for c in criteria.keys() if c not in summary.columns]
if missing:
    raise KeyError(
        "Missing required columns in summary:\n"
        + "\n".join(missing)
        + f"\n\nTip: check MODEL='{MODEL}' matches your column prefixes."
    )

criteria_cols = list(criteria.keys())

# ---------------------------------------------------------
# Exclude NaN sessions from ALL statistics (any NaN in criteria cols)
# ---------------------------------------------------------
valid_rows = summary[criteria_cols].notna().all(axis=1)
summary_valid = summary.loc[valid_rows].copy()

n_excluded_nan = int((~valid_rows).sum())
print(f"Excluded {n_excluded_nan} sessions due to NaN in criteria columns.")

# ---------------------------------------------------------
# Build mask (pass all criteria) on valid data only
# ---------------------------------------------------------
mask = np.ones(len(summary_valid), dtype=bool)
for col, (fn, _) in criteria.items():
    mask &= summary_valid[col].apply(lambda v: bool(fn(v)))

satisfied_sessions = summary_valid[mask]
not_satisfied_sessions = summary_valid[~mask]

n_all = len(summary_valid)
n_failed = len(not_satisfied_sessions)

# ---------------------------------------------------------
# Per-criterion failure stats (valid rows only)
# ---------------------------------------------------------
failure_counts_all = {}
failure_counts_failed = {}

for col, (fn, msg) in criteria.items():
    fail_mask = ~summary_valid[col].apply(lambda v: bool(fn(v)))  # no NaNs here by construction
    failure_counts_all[msg] = int(fail_mask.sum())

    if n_failed > 0:
        fail_mask_failed = fail_mask & (~mask)
        failure_counts_failed[msg] = int(fail_mask_failed.sum())
    else:
        failure_counts_failed[msg] = 0

failure_stats = (
    pd.DataFrame({
        "criterion": list(failure_counts_all.keys()),
        "n_fail_all": list(failure_counts_all.values()),
        "frac_all": [c / n_all if n_all else np.nan for c in failure_counts_all.values()],
        "n_fail_among_failed": list(failure_counts_failed.values()),
        "frac_among_failed": [c / n_failed if n_failed else np.nan for c in failure_counts_failed.values()],
    })
    .sort_values(["frac_among_failed", "frac_all", "n_fail_among_failed"], ascending=False)
    .reset_index(drop=True)
)

print(f"MODEL = {MODEL}")
print(f"Num sessions total (valid only): {n_all}")
print(f"Num satisfied: {len(satisfied_sessions)}")
print(f"Num not satisfied: {n_failed}")

try:
    from IPython.display import display
    display(failure_stats)
except Exception:
    pass


Excluded 234 sessions due to NaN in criteria columns.
MODEL = QLearning_L1F1_CK1_softmax
Num sessions total (valid only): 9493
Num satisfied: 4507
Num not satisfied: 4986


,criterion,n_fail_all,frac_all,n_fail_among_failed,frac_among_failed
0,Abs(Bias L) >= 0.5,2783,0.293163,2783,0.558163
1,Learn Rate <= 0.15,1866,0.196566,1866,0.374248
2,LPT <= 0.65,1405,0.148004,1405,0.281789
3,Reward Coefs Sum6 >= 0.8,703,0.074055,703,0.140995
4,SumQ Std <= 0.15,334,0.035184,334,0.066988
5,DeltaQ Std <= 0.2,223,0.023491,223,0.044725
6,DeltaQ Entropy <= 2.5,13,0.001369,13,0.002607
7,RPE Entropy <= 2.25,0,0.000000,0,0.000000
8,RPE Std <= 0.45,0,0.000000,0,0.000000


In [22]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Configure which model's columns you want to filter on
# ---------------------------------------------------------


# ---------------------------------------------------------
# Define criteria (value must PASS the lambda to be satisfied)
# NOTE: We will EXCLUDE any session with NaN in ANY criteria column
# ---------------------------------------------------------

# ---------------------------------------------------------
# Sanity check required columns exist
# ---------------------------------------------------------
missing = [c for c in criteria.keys() if c not in summary.columns]
if missing:
    raise KeyError(
        "Missing required columns in summary:\n"
        + "\n".join(missing)
        + f"\n\nTip: check MODEL='{MODEL}' matches your column prefixes."
    )

criteria_cols = list(criteria.keys())

# ---------------------------------------------------------
# Exclude non-valid sessions: any NaN in ANY criteria column
# ---------------------------------------------------------
valid_mask = summary[criteria_cols].notna().all(axis=1)
summary_valid = summary.loc[valid_mask].copy()

n_excluded = int((~valid_mask).sum())
print(f"Excluded {n_excluded} sessions due to NaN in criteria columns.")

# ---------------------------------------------------------
# Build per-criterion fail masks on VALID data only
# (No NaNs remain in these columns)
# ---------------------------------------------------------
fail_masks = {}
for col, (fn, msg) in criteria.items():
    fail_masks[msg] = ~summary_valid[col].apply(lambda v: bool(fn(v)))

# Overall pass/fail mask (must pass ALL)
mask_pass_all = np.ones(len(summary_valid), dtype=bool)
for msg, fmask in fail_masks.items():
    mask_pass_all &= (~fmask)

satisfied_sessions = summary_valid[mask_pass_all]
not_satisfied_sessions = summary_valid[~mask_pass_all]

n_all = len(summary_valid)
n_satisfied = len(satisfied_sessions)
n_failed = len(not_satisfied_sessions)

# ---------------------------------------------------------
# Overall satisfaction fraction among valid sessions
# (satisfied / (satisfied + not satisfied)) == satisfied / valid
# ---------------------------------------------------------
frac_satisfied = (n_satisfied / (n_satisfied + n_failed)) if (n_satisfied + n_failed) else np.nan

# ---------------------------------------------------------
# 1) Single-filter failure stats (sorted by FRACTIONS)
# ---------------------------------------------------------
failure_stats_1 = (
    pd.DataFrame({
        "criterion": list(fail_masks.keys()),
        "n_fail_all": [int(fail_masks[msg].sum()) for msg in fail_masks.keys()],
        "frac_all": [float(fail_masks[msg].mean()) if n_all else np.nan for msg in fail_masks.keys()],
        "n_fail_among_failed": [int((fail_masks[msg] & (~mask_pass_all)).sum()) for msg in fail_masks.keys()],
        "frac_among_failed": [
            (int((fail_masks[msg] & (~mask_pass_all)).sum()) / n_failed) if n_failed else np.nan
            for msg in fail_masks.keys()
        ],
    })
    .sort_values(
        by=["frac_among_failed", "frac_all", "n_fail_among_failed"],
        ascending=False
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# 2) Pairwise stats: FAIL EITHER (A OR B) and FAIL BOTH (A AND B)
#    Sorted by FRACTIONS (value-based)
# ---------------------------------------------------------
msgs = list(fail_masks.keys())
pair_rows = []

for i in range(len(msgs)):
    for j in range(i + 1, len(msgs)):
        a, b = msgs[i], msgs[j]

        fail_a = fail_masks[a]
        fail_b = fail_masks[b]

        fail_either = fail_a | fail_b          # fail at least one of the two
        fail_both = fail_a & fail_b            # fail both (satisfy neither)

        n_fail_either_all = int(fail_either.sum())
        n_fail_both_all = int(fail_both.sum())

        n_fail_either_among_failed = int((fail_either & (~mask_pass_all)).sum())
        n_fail_both_among_failed = int((fail_both & (~mask_pass_all)).sum())

        pair_rows.append({
            "criterion_A": a,
            "criterion_B": b,

            "n_fail_either_all": n_fail_either_all,
            "frac_fail_either_all": (n_fail_either_all / n_all) if n_all else np.nan,

            "n_fail_either_among_failed": n_fail_either_among_failed,
            "frac_fail_either_among_failed": (n_fail_either_among_failed / n_failed) if n_failed else np.nan,

            "n_fail_both_all": n_fail_both_all,
            "frac_fail_both_all": (n_fail_both_all / n_all) if n_all else np.nan,

            "n_fail_both_among_failed": n_fail_both_among_failed,
            "frac_fail_both_among_failed": (n_fail_both_among_failed / n_failed) if n_failed else np.nan,
        })

failure_stats_2 = (
    pd.DataFrame(pair_rows)
    .sort_values(
        by=[
            "frac_fail_either_among_failed",
            "frac_fail_both_among_failed",
            "frac_fail_either_all",
            "frac_fail_both_all",
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------
print(f"MODEL = {MODEL}")
print(f"Num sessions total (valid only): {n_all}")
print(f"Num satisfied (pass all): {n_satisfied}")
print(f"Num not satisfied (fail >=1): {n_failed}")
print(f"Frac satisfied (satisfied / (satisfied + not satisfied)): {frac_satisfied:.6f}" if np.isfinite(frac_satisfied) else "Frac satisfied: NaN")

# Notebook-friendly display (works even if stdout print is broken)
try:
    from IPython.display import display
    print("\nSingle-filter failure stats (sorted by fractions):")
    display(failure_stats_1)

    print("\nTwo-filter (pairwise) stats (sorted by fractions):")
    print("  - fail_either: fail at least one of the two")
    print("  - fail_both:   fail both (satisfy neither)")
    display(failure_stats_2)
except Exception:
    pass


Excluded 234 sessions due to NaN in criteria columns.
MODEL = QLearning_L1F1_CK1_softmax
Num sessions total (valid only): 9493
Num satisfied (pass all): 4507
Num not satisfied (fail >=1): 4986
Frac satisfied (satisfied / (satisfied + not satisfied)): 0.474771

Single-filter failure stats (sorted by fractions):


,criterion,n_fail_all,frac_all,n_fail_among_failed,frac_among_failed
0,Abs(Bias L) >= 0.5,2783,0.293163,2783,0.558163
1,Learn Rate <= 0.15,1866,0.196566,1866,0.374248
2,LPT <= 0.65,1405,0.148004,1405,0.281789
3,Reward Coefs Sum6 >= 0.8,703,0.074055,703,0.140995
4,SumQ Std <= 0.15,334,0.035184,334,0.066988
5,DeltaQ Std <= 0.2,223,0.023491,223,0.044725
6,DeltaQ Entropy <= 2.5,13,0.001369,13,0.002607
7,RPE Entropy <= 2.25,0,0.000000,0,0.000000
8,RPE Std <= 0.45,0,0.000000,0,0.000000



Two-filter (pairwise) stats (sorted by fractions):
  - fail_either: fail at least one of the two
  - fail_both:   fail both (satisfy neither)


,criterion_A,criterion_B,n_fail_either_all,frac_fail_either_all,n_fail_either_among_failed,frac_fail_either_among_failed,n_fail_both_all,frac_fail_both_all,n_fail_both_among_failed,frac_fail_both_among_failed
0,Learn Rate <= 0.15,Abs(Bias L) >= 0.5,4297,0.452649,4297,0.861813,352,0.037080,352,0.070598
1,Abs(Bias L) >= 0.5,LPT <= 0.65,3947,0.415780,3947,0.791617,241,0.025387,241,0.048335
2,Abs(Bias L) >= 0.5,Reward Coefs Sum6 >= 0.8,3410,0.359212,3410,0.683915,76,0.008006,76,0.015243
3,Abs(Bias L) >= 0.5,SumQ Std <= 0.15,3074,0.323818,3074,0.616526,43,0.004530,43,0.008624
4,Abs(Bias L) >= 0.5,DeltaQ Std <= 0.2,2972,0.313073,2972,0.596069,34,0.003582,34,0.006819
5,Abs(Bias L) >= 0.5,DeltaQ Entropy <= 2.5,2791,0.294006,2791,0.559767,5,0.000527,5,0.001003
6,Abs(Bias L) >= 0.5,RPE Entropy <= 2.25,2783,0.293163,2783,0.558163,0,0.000000,0,0.000000
7,Abs(Bias L) >= 0.5,RPE Std <= 0.45,2783,0.293163,2783,0.558163,0,0.000000,0,0.000000
8,Learn Rate <= 0.15,LPT <= 0.65,2727,0.287264,2727,0.546931,544,0.057305,544,0.109105
9,Learn Rate <= 0.15,Reward Coefs Sum6 >= 0.8,1867,0.196671,1867,0.374448,702,0.073949,702,0.140794


In [24]:
import numpy as np
import pandas as pd
from itertools import combinations

# ---------------------------------------------------------
# Configure which model's columns you want to filter on
# ---------------------------------------------------------


# ---------------------------------------------------------
# Define criteria (value must PASS the lambda to be satisfied)
# NOTE: We will EXCLUDE any session with NaN in ANY criteria column
# ---------------------------------------------------------


# ---------------------------------------------------------
# Sanity check required columns exist
# ---------------------------------------------------------
missing = [c for c in criteria.keys() if c not in summary.columns]
if missing:
    raise KeyError(
        "Missing required columns in summary:\n"
        + "\n".join(missing)
        + f"\n\nTip: check MODEL='{MODEL}' matches your column prefixes."
    )

criteria_cols = list(criteria.keys())

# ---------------------------------------------------------
# Exclude non-valid sessions: any NaN in ANY criteria column
# ---------------------------------------------------------
valid_mask = summary[criteria_cols].notna().all(axis=1)
summary_valid = summary.loc[valid_mask].copy()

n_excluded = int((~valid_mask).sum())
print(f"Excluded {n_excluded} sessions due to NaN in criteria columns.")

# ---------------------------------------------------------
# Build per-criterion fail masks on VALID data only (no NaNs)
# ---------------------------------------------------------
fail_masks = {}
for col, (fn, msg) in criteria.items():
    fail_masks[msg] = ~summary_valid[col].apply(lambda v: bool(fn(v)))

# Overall pass/fail mask (must pass ALL)
mask_pass_all = np.ones(len(summary_valid), dtype=bool)
for msg, fmask in fail_masks.items():
    mask_pass_all &= (~fmask)

satisfied_sessions = summary_valid[mask_pass_all]
not_satisfied_sessions = summary_valid[~mask_pass_all]

n_all = len(summary_valid)
n_failed = len(not_satisfied_sessions)

# ---------------------------------------------------------
# 3-criteria stats: FAIL EITHER (A OR B OR C)
#   i.e., fail at least one of the three criteria
# ---------------------------------------------------------
msgs = list(fail_masks.keys())
triplet_rows = []

for a, b, c in combinations(msgs, 3):
    fail_any = fail_masks[a] | fail_masks[b] | fail_masks[c]

    n_fail_any_all = int(fail_any.sum())
    n_fail_any_among_failed = int((fail_any & (~mask_pass_all)).sum())

    triplet_rows.append({
        "criterion_A": a,
        "criterion_B": b,
        "criterion_C": c,
        "n_fail_any_all": n_fail_any_all,
        "frac_fail_any_all": (n_fail_any_all / n_all) if n_all else np.nan,
        "n_fail_any_among_failed": n_fail_any_among_failed,
        "frac_fail_any_among_failed": (n_fail_any_among_failed / n_failed) if n_failed else np.nan,
    })

failure_stats_3 = (
    pd.DataFrame(triplet_rows)
    .sort_values("frac_fail_any_among_failed", ascending=False)  # largest first (as requested)
    .reset_index(drop=True)
)

print(f"MODEL = {MODEL}")
print(f"Num sessions total (valid only): {n_all}")
print(f"Num satisfied (pass all): {len(satisfied_sessions)}")
print(f"Num not satisfied (fail >=1): {n_failed}")

# Notebook-friendly display
try:
    from IPython.display import display
    print("\nThree-filter (triplet) stats: fail ANY of the three (A OR B OR C)")
    display(failure_stats_3)
except Exception:
    pass


Excluded 234 sessions due to NaN in criteria columns.
MODEL = QLearning_L1F1_CK1_softmax
Num sessions total (valid only): 9493
Num satisfied (pass all): 4507
Num not satisfied (fail >=1): 4986

Three-filter (triplet) stats: fail ANY of the three (A OR B OR C)


,criterion_A,criterion_B,criterion_C,n_fail_any_all,frac_fail_any_all,n_fail_any_among_failed,frac_fail_any_among_failed
0,Learn Rate <= 0.15,Abs(Bias L) >= 0.5,LPT <= 0.65,4986,0.525229,4986,1.000000
1,Abs(Bias L) >= 0.5,LPT <= 0.65,Reward Coefs Sum6 >= 0.8,4305,0.453492,4305,0.863418
2,Learn Rate <= 0.15,Abs(Bias L) >= 0.5,DeltaQ Entropy <= 2.5,4297,0.452649,4297,0.861813
3,Learn Rate <= 0.15,Abs(Bias L) >= 0.5,Reward Coefs Sum6 >= 0.8,4297,0.452649,4297,0.861813
4,Learn Rate <= 0.15,Abs(Bias L) >= 0.5,DeltaQ Std <= 0.2,4297,0.452649,4297,0.861813
...,...,...,...,...,...,...,...
79,DeltaQ Entropy <= 2.5,SumQ Std <= 0.15,RPE Std <= 0.45,334,0.035184,334,0.066988
80,DeltaQ Entropy <= 2.5,RPE Entropy <= 2.25,DeltaQ Std <= 0.2,223,0.023491,223,0.044725
81,DeltaQ Entropy <= 2.5,DeltaQ Std <= 0.2,RPE Std <= 0.45,223,0.023491,223,0.044725
82,RPE Entropy <= 2.25,DeltaQ Std <= 0.2,RPE Std <= 0.45,223,0.023491,223,0.044725
